# Ingredient substitution: data loading and baseline methods

This notebook

1. downloads the five recipe sources and combines them into one pandas DataFrame,
2. normalises ingredient names and removes duplicate recipes,
3. implements four baseline substitution methods behind one interface:
   popularity, co-occurrence similarity, association rules (mlxtend) and Word2Vec (gensim),
4. fits each method globally and per cuisine, and runs a Hit@k / MRR evaluation on a placeholder pair list.

**Before running**

* Kaggle credentials: put `kaggle.json` in `~/.kaggle/` or set `KAGGLE_USERNAME` and `KAGGLE_KEY`.
* *What's Cooking?* is a competition dataset. You must accept the competition rules on the Kaggle website once, otherwise the download returns 403.
* RecipeNLG is about 2.2M recipes (roughly 2 GB unzipped). It is sampled by default, see `MAX_RECIPES`.
* Only *What's Cooking?* has cuisine labels. Food.com cuisine labels are derived from tags/keywords below. Eight Portions and RecipeNLG have no cuisine, so they only contribute to the global models.

In [ ]:
%pip install -q pandas numpy scipy requests tqdm matplotlib kagglehub mlxtend gensim

In [ ]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

In [ ]:
import ast
import inspect
import json
import random
import re
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import sparse

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Maximum number of recipes kept per source (None = keep all).
MAX_RECIPES = {
    "whats_cooking": None,
    "eightportions": None,
    "recipenlg": None, #300_000,
    "foodcom_irkaal": None,
    "foodcom_shuyangli94": None,
}

MIN_INGREDIENT_COUNT = 5        # drop ingredients that appear in fewer recipes than this
MIN_RECIPES_PER_CUISINE = 300   # skip per-cuisine models below this size

## 1. Download and load each source

Every loader returns the same columns: `source`, `source_id`, `title`, `cuisine`, `ingredients_raw` (list of strings).

In [ ]:
def extract_zips(root):
    # Extract every zip under root into <name>_unzipped (skips if already extracted).
    root = Path(root)
    for z in list(root.rglob("*.zip")):
        out = z.parent / (z.name[:-4] + "_unzipped")
        if not out.exists():
            with zipfile.ZipFile(z) as f:
                f.extractall(out)
    return root


def find_file(root, pattern):
    hits = sorted(p for p in Path(root).rglob(pattern) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"No file matching {pattern!r} under {root}")
    return hits[0]


def download(url, dest):
    dest = Path(dest)
    if dest.exists():
        return dest
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    return dest


def cap(df, n):
    return df if n is None or len(df) <= n else df.sample(n=n, random_state=SEED)


_R_STRING = re.compile(r'"((?:[^"\\]|\\.)*)"')

def parse_list(value):
    # Parse list-like cells: Python/JSON list strings, R c("a", "b") vectors, arrays.
    if isinstance(value, (list, tuple, np.ndarray)):
        return [str(v) for v in value if v is not None]
    if value is None or pd.isna(value):
        return []
    s = str(value).strip()
    if not s or s in {"NA", "character(0)"}:
        return []
    if s.startswith("["):
        try:
            return [str(v) for v in json.loads(s)]
        except ValueError:
            try:
                return [str(v) for v in ast.literal_eval(s)]
            except (ValueError, SyntaxError):
                pass
    if s.startswith("c(") or s.startswith('"'):
        return _R_STRING.findall(s)
    return [s]


# Map Food.com tags / keywords onto the 20 What's Cooking cuisines.
# Check these against the actual tag value counts before trusting them.
CUISINE_TAGS = {
    "greek": {"greek"},
    "southern_us": {"southern", "southern-united-states", "southern united states", "soul"},
    "filipino": {"filipino", "philippine"},
    "indian": {"indian"},
    "jamaican": {"jamaican"},
    "spanish": {"spanish"},
    "italian": {"italian"},
    "mexican": {"mexican"},
    "chinese": {"chinese"},
    "british": {"british", "english", "scottish", "welsh"},
    "thai": {"thai"},
    "vietnamese": {"vietnamese"},
    "cajun_creole": {"cajun", "creole"},
    "brazilian": {"brazilian"},
    "french": {"french"},
    "japanese": {"japanese"},
    "irish": {"irish"},
    "korean": {"korean"},
    "moroccan": {"moroccan"},
    "russian": {"russian"},
}

def cuisine_from_tags(tags):
    # Return a cuisine only if exactly one cuisine matches, otherwise None.
    tags = {t.strip().lower() for t in tags}
    hits = [c for c, keys in CUISINE_TAGS.items() if tags & keys]
    return hits[0] if len(hits) == 1 else None

In [ ]:
EIGHTPORTIONS_URL = "https://eightportions.com/recipes_raw.zip"

def load_eightportions():
    root = DATA_DIR / "eightportions"
    root.mkdir(exist_ok=True)
    download(EIGHTPORTIONS_URL, root / "recipes_raw.zip")
    extract_zips(root)
    site = {"ar": "allrecipes", "epi": "epicurious", "fn": "foodnetwork"}
    rows = []
    for f in sorted(root.rglob("recipes_raw*.json")):
        tag = f.stem.split("_")[-1]
        with open(f, encoding="utf-8") as fh:
            data = json.load(fh)
        for rid, rec in data.items():
            if not rec:
                continue
            # Allrecipes ingredient strings carry a scraped "ADVERTISEMENT" marker.
            ings = [i.replace("ADVERTISEMENT", "").strip() for i in (rec.get("ingredients") or [])]
            rows.append({
                "source": "eightportions_" + site.get(tag, tag),
                "source_id": f"ep_{tag}:{rid}",
                "title": rec.get("title"),
                "cuisine": None,
                "ingredients_raw": [i for i in ings if i],
            })
    return pd.DataFrame(rows)


RECIPENLG_TOTAL = 2_231_142  # rows in the full file, used to pick a sampling fraction

def load_recipenlg(n=MAX_RECIPES["recipenlg"]):
    import kagglehub
    root = extract_zips(kagglehub.dataset_download("paultimothymooney/recipenlg"))
    path = find_file(root, "*.csv")
    frac = 1.0 if n is None else min(1.0, 1.2 * n / RECIPENLG_TOTAL)
    parts = []
    for chunk in pd.read_csv(path, usecols=["title", "NER", "link"], chunksize=200_000):
        parts.append(chunk if frac >= 1 else chunk.sample(frac=frac, random_state=SEED))
    df = cap(pd.concat(parts, ignore_index=True), n)
    # NER holds the cleaned ingredient names, which avoids parsing quantities.
    return pd.DataFrame({
        "source": "recipenlg",
        "source_id": "nlg:" + df["link"].astype(str),
        "title": df["title"],
        "cuisine": None,
        "ingredients_raw": df["NER"].map(parse_list),
    })


def load_foodcom_irkaal():
    import kagglehub
    root = extract_zips(kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews"))
    cols = ["RecipeId", "Name", "Keywords", "RecipeIngredientParts"]
    try:
        df = pd.read_parquet(find_file(root, "recipes.parquet"), columns=cols)
    except (FileNotFoundError, ImportError):
        df = pd.read_csv(find_file(root, "recipes.csv"), usecols=cols)
    ids = pd.to_numeric(df["RecipeId"], errors="coerce").astype("Int64").astype(str)
    return pd.DataFrame({
        "source": "foodcom_irkaal",
        "source_id": "foodcom:" + ids,   # same id space as the shuyangli94 Food.com dump
        "title": df["Name"],
        "cuisine": df["Keywords"].map(parse_list).map(cuisine_from_tags),
        "ingredients_raw": df["RecipeIngredientParts"].map(parse_list),
    })


def load_foodcom_shuyangli94():
    import kagglehub
    root = extract_zips(kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions"))

    # excluded but potentially useful: Text for recipe steps, in order
    df = pd.read_csv(find_file(root, "RAW_recipes.csv"), usecols=["id", "name", "tags", "ingredients"])
    df.prin
    return pd.DataFrame({
        "source": "foodcom_shuyangli94",
        "source_id": "foodcom:" + df["id"].astype(str),
        "title": df["name"],
        "cuisine": df["tags"].map(parse_list).map(cuisine_from_tags),
        "ingredients_raw": df["ingredients"].map(parse_list),
    })

In [ ]:
LOADERS = {
    "eightportions": load_eightportions,
    "recipenlg": load_recipenlg,
    "foodcom_irkaal": load_foodcom_irkaal,
    "foodcom_shuyangli94": load_foodcom_shuyangli94,
}

frames = {}
for name, loader in LOADERS.items():
    try:
        frames[name] = cap(loader(), MAX_RECIPES[name])
        print(f"{name:22s} {len(frames[name]):>9,} recipes")
    except Exception as e:  # one failing source should not stop the others
        print(f"[skipped] {name}: {type(e).__name__}: {e}")

raw = pd.concat(frames.values(), ignore_index=True)
raw.groupby("source").size()

eightportions            124,647 recipes
Using Colab cache for faster access to the 'recipenlg' dataset.
recipenlg              2,231,142 recipes
Using Colab cache for faster access to the 'foodcom-recipes-and-reviews' dataset.
foodcom_irkaal           522,517 recipes
Using Colab cache for faster access to the 'food-com-recipes-and-user-interactions' dataset.
foodcom_shuyangli94      231,637 recipes


,0
source,
eightportions_allrecipes,39522
eightportions_epicurious,25323
eightportions_foodnetwork,59802
foodcom_irkaal,522517
foodcom_shuyangli94,231637
recipenlg,2231142


In [ ]:
raw

,source,source_id,title,cuisine,ingredients_raw
0,eightportions_allrecipes,ep_ar:rmK12Uau.ntP510KeImX506H6Mr6jTu,Slow Cooker Chicken and Dumplings,None,"[4 skinless, boneless chicken breast halves, 2..."
1,eightportions_allrecipes,ep_ar:5ZpZE8hSVdPk2ZXo1mZTyoPWJRSCPSm,Awesome Slow Cooker Pot Roast,None,[2 (10.75 ounce) cans condensed cream of mushr...
2,eightportions_allrecipes,ep_ar:clyYQv.CplpwJtjNaFGhx0VilNYqRxu,Brown Sugar Meatloaf,None,"[1/2 cup packed brown sugar, 1/2 cup ketchup, ..."
3,eightportions_allrecipes,ep_ar:BmqFAmCrDHiKNwX.IQzb0U/v0mLlxFu,Best Chocolate Chip Cookies,None,"[1 cup butter, softened, 1 cup white sugar, 1 ..."
4,eightportions_allrecipes,ep_ar:N.jCksRjB4MFwbgPFQU8Kg.yF.XCtOi,Homemade Mac and Cheese Casserole,None,"[8 ounces whole wheat rotini pasta, 3 cups fre..."
...,...,...,...,...,...
3109938,foodcom_shuyangli94,foodcom:486161,zydeco soup,None,"[celery, onion, green sweet pepper, garlic clo..."
3109939,foodcom_shuyangli94,foodcom:493372,zydeco spice mix,None,"[paprika, salt, garlic powder, onion powder, d..."
3109940,foodcom_shuyangli94,foodcom:308080,zydeco ya ya deviled eggs,None,"[hard-cooked eggs, mayonnaise, dijon mustard, ..."
3109941,foodcom_shuyangli94,foodcom:298512,cookies by design cookies on a stick,None,"[butter, eagle brand condensed milk, light bro..."


## 2. Normalise ingredient names and deduplicate

The sources differ a lot: What's Cooking and RecipeNLG `NER` give bare names, Food.com gives mostly bare names,
Eight Portions gives full lines such as `"2 cups all-purpose flour, sifted"`. The normaliser below is rule based and crude
(units, quantities, preparation words, plurals). Inspect the most frequent outputs before relying on it.

In [ ]:
UNITS = {
    "cup", "cups", "c", "tablespoon", "tablespoons", "tbsp", "tbs", "tbl", "teaspoon", "teaspoons", "tsp",
    "ounce", "ounces", "oz", "pound", "pounds", "lb", "lbs", "gram", "grams", "g", "kg", "kilogram", "kilograms",
    "ml", "milliliter", "milliliters", "l", "liter", "liters", "litre", "litres", "quart", "quarts", "qt",
    "pint", "pints", "pt", "gallon", "gallons", "pinch", "pinches", "dash", "dashes", "can", "cans",
    "package", "packages", "pkg", "jar", "jars", "bottle", "bottles", "bunch", "bunches", "envelope", "envelopes",
}
PREP_WORDS = {
    "chopped", "minced", "diced", "sliced", "crushed", "grated", "shredded", "peeled", "seeded", "cubed",
    "softened", "melted", "divided", "packed", "sifted", "beaten", "drained", "rinsed", "trimmed", "halved",
    "quartered", "finely", "coarsely", "roughly", "thinly", "freshly", "fresh", "large", "small", "medium",
    "optional", "taste", "to", "of", "and", "or", "for", "about", "plus", "more", "needed", "as", "a", "an",
}
NO_SINGULAR = {"molasses", "asparagus", "hummus", "couscous", "swiss", "grits", "brussels", "citrus",
               "octopus", "schnapps", "bass", "lemongrass", "watercress", "hibiscus", "anise", "cheese"}

IRREGULAR = {"leaves": "leaf", "halves": "half", "loaves": "loaf", "knives": "knife"}

def singular(tok):
    if tok in IRREGULAR:
        return IRREGULAR[tok]
    if tok in NO_SINGULAR or tok.endswith(("ss", "us")) or len(tok) <= 3:
        return tok
    if tok.endswith("ies"):
        return tok[:-3] + "y"
    if tok.endswith(("oes", "ches", "shes")):
        return tok[:-2]
    if tok.endswith("s"):
        return tok[:-1]
    return tok

def normalize_ingredient(s):
    s = str(s).lower()
    s = re.sub(r"\(.*?\)", " ", s)            # drop parentheticals: "(14 ounce)"
    s = s.split(",")[0]                         # "onion, chopped" -> "onion"
    tokens = re.findall(r"[a-z]+(?:[-'][a-z]+)*", s)
    if "garlic" in tokens:                      # "cloves garlic" -> "garlic", but keep the spice "cloves"
        tokens = [t for t in tokens if t not in {"clove", "cloves"}]
    tokens = [singular(t) for t in tokens if t not in UNITS and t not in PREP_WORDS]
    return " ".join(tokens)

for example in ["2 cups all-purpose flour, sifted", "3 cloves garlic, minced", "1/2 tsp ground cloves",
                "1 (14 ounce) can diced tomatoes", "Half-and-Half", "fresh cilantro leaves", "2 bay leaves"]:
    print(f"{example!r:40} -> {normalize_ingredient(example)!r}")

'2 cups all-purpose flour, sifted'       -> 'all-purpose flour'
'3 cloves garlic, minced'                -> 'garlic'
'1/2 tsp ground cloves'                  -> 'ground clove'
'1 (14 ounce) can diced tomatoes'        -> 'tomato'
'Half-and-Half'                          -> 'half-and-half'
'fresh cilantro leaves'                  -> 'cilantro leaf'
'2 bay leaves'                           -> 'bay leaf'


In [ ]:
raw["ingredients"] = raw["ingredients_raw"].map(
    lambda xs: sorted({n for n in map(normalize_ingredient, xs) if n})
)

# 1) The two Food.com dumps share recipe ids: keep one copy, preferring a row with a cuisine label.
recipes = (raw.assign(_has_cuisine=raw["cuisine"].notna())
              .sort_values("_has_cuisine", ascending=False)
              .drop_duplicates("source_id")
              .drop(columns="_has_cuisine"))

# 2) The same recipe republished across sources (e.g. RecipeNLG vs Eight Portions vs Food.com):
#    same title and same normalised ingredient set. Untitled What's Cooking rows are never dropped here.
key = recipes["title"].fillna("").str.lower().str.strip() + "|" + recipes["ingredients"].map("|".join)
has_title = recipes["title"].fillna("").str.strip().ne("")
recipes = recipes[~(key.duplicated() & has_title)]

# 3) Remove rare ingredients, then recipes with fewer than two ingredients.
ing_counts = Counter(i for r in recipes["ingredients"] for i in r)
keep = {i for i, c in ing_counts.items() if c >= MIN_INGREDIENT_COUNT}
recipes = recipes.assign(ingredients=recipes["ingredients"].map(lambda r: [i for i in r if i in keep]))
recipes = recipes[recipes["ingredients"].map(len) >= 2].reset_index(drop=True)

print(f"{len(raw):,} raw rows -> {len(recipes):,} recipes, {len(keep):,} ingredients")
print(f"recipes with a cuisine label: {recipes['cuisine'].notna().mean():.1%}")
display(recipes.groupby("source").size().rename("recipes").to_frame())
display(recipes["cuisine"].value_counts().rename("recipes").to_frame())
print(pd.Series(Counter(i for r in recipes["ingredients"] for i in r)).nlargest(30))

3,109,943 raw rows -> 2,721,199 recipes, 37,532 ingredients
recipes with a cuisine label: 2.1%


,recipes
source,
eightportions_allrecipes,38882
eightportions_epicurious,24373
eightportions_foodnetwork,54418
foodcom_irkaal,325817
foodcom_shuyangli94,175022
recipenlg,2102687


,recipes
cuisine,
mexican,13103
italian,7122
indian,5941
southern_us,5171
greek,3867
chinese,3809
cajun_creole,3117
british,2695
thai,2203


salt                   1127879
egg                     681121
sugar                   665886
butter                  606899
onion                   593982
garlic                  558619
flour                   489412
water                   411560
milk                    397199
olive oil               300804
vanilla                 284441
tomato                  244906
pepper                  242049
brown sugar             216958
lemon juice             197897
baking powder           192704
parsley                 156288
carrot                  154014
baking soda             148845
sour cream              144933
cinnamon                143076
cream cheese            137332
celery                  135303
cheddar cheese          124929
parmesan cheese         124326
margarine               120206
vegetable oil           118477
chicken                 115382
ground black pepper     113503
mayonnaise              110311
dtype: int64


In [ ]:
recipes.to_pickle(DATA_DIR / "recipes_combined.pkl")

## 3. Methods

All four methods share the interface `fit(recipes)` and `rank(query, k)`, which returns `(ingredient, score)` pairs, best first.

A point that matters for the write-up: **substitutes rarely appear in the same recipe** (butter and margarine), while
**complements do** (butter and flour). Ranking by direct co-occurrence or by rule lift therefore finds complements.
So the co-occurrence and association-rule models default to a *second-order* score (two ingredients are similar if they
co-occur with the same other ingredients). The first-order version is still available with `mode="direct"` for comparison.

In [ ]:
def build_matrix(recipe_lists, vocab=None):
    # Binary recipe x ingredient CSR matrix.
    if vocab is None:
        vocab = sorted({i for r in recipe_lists for i in r})
    idx = {w: j for j, w in enumerate(vocab)}
    rows, cols = [], []
    for r_i, r in enumerate(recipe_lists):
        for ing in set(r):
            j = idx.get(ing)
            if j is not None:
                rows.append(r_i)
                cols.append(j)
    X = sparse.csr_matrix((np.ones(len(rows), dtype=np.float64), (rows, cols)),
                          shape=(len(recipe_lists), len(vocab)))
    return X, list(vocab), idx


def top_k(scores, vocab, k, exclude):
    scores = np.asarray(scores, dtype=float).copy()
    scores[exclude] = -np.inf
    k = min(k, len(scores) - 1)
    if k <= 0:
        return []
    cand = np.argpartition(-scores, k - 1)[:k]
    cand = cand[np.argsort(-scores[cand])]
    return [(vocab[j], float(scores[j])) for j in cand if np.isfinite(scores[j]) and scores[j] > 0]


class SubstitutionModel:
    name = "base"
    idx = {}

    def knows(self, ingredient):
        return ingredient in self.idx

    def fit(self, recipe_lists):
        raise NotImplementedError

    def rank(self, query, k=10):
        raise NotImplementedError

### 3.1 Popularity baseline

Ranks every ingredient by how many recipes it appears in. The list is the same for every query, which is what makes it a floor.

In [ ]:
class PopularityBaseline(SubstitutionModel):
    name = "popularity"

    def fit(self, recipe_lists):
        self.counts = Counter(i for r in recipe_lists for i in set(r))
        self.ranking = [w for w, _ in self.counts.most_common()]
        self.idx = {w: j for j, w in enumerate(self.ranking)}
        return self

    def rank(self, query, k=10):
        out = []
        for w in self.ranking:
            if w != query:
                out.append((w, float(self.counts[w])))
                if len(out) == k:
                    break
        return out

### 3.2 Co-occurrence similarity

Builds the ingredient co-occurrence matrix, converts it to positive PMI, and scores candidates either by
cosine similarity of their PPMI rows (`mode="context"`, default) or by PPMI with the query (`mode="direct"`).

In [ ]:
class CooccurrenceSimilarity(SubstitutionModel):
    def __init__(self, mode="context", min_cooc=2):
        assert mode in {"context", "direct"}
        self.mode, self.min_cooc = mode, min_cooc
        self.name = f"cooccurrence_{mode}"

    def fit(self, recipe_lists):
        X, self.vocab, self.idx = build_matrix(recipe_lists)
        n = X.shape[0]
        doc_freq = np.asarray(X.sum(axis=0)).ravel()
        C = (X.T @ X).tocoo()
        keep = (C.row != C.col) & (C.data >= self.min_cooc)
        r, c, v = C.row[keep], C.col[keep], C.data[keep]
        pmi = np.log(v * n / (doc_freq[r] * doc_freq[c]))
        pos = pmi > 0
        V = len(self.vocab)
        self.ppmi = sparse.csr_matrix((pmi[pos], (r[pos], c[pos])), shape=(V, V))
        norms = np.sqrt(np.asarray(self.ppmi.multiply(self.ppmi).sum(axis=1)).ravel())
        norms[norms == 0] = 1.0
        self.ppmi_unit = sparse.csr_matrix(sparse.diags(1.0 / norms) @ self.ppmi)
        return self

    def rank(self, query, k=10):
        q = self.idx[query]
        if self.mode == "direct":
            scores = self.ppmi[q].toarray().ravel()
        else:
            scores = (self.ppmi_unit @ self.ppmi_unit[q].T).toarray().ravel()
        return top_k(scores, self.vocab, k, exclude=q)

### 3.3 Association rules (mlxtend)

Mines pairwise rules `{a} -> {b}` with FP-growth on the most frequent ingredients.
`mode="direct"` ranks the consequents of the query by lift (complements).
`mode="context"` (default) represents each ingredient by the confidence of its rules to all consequents and compares
these profiles with cosine similarity.

Note: for single-item rules, lift equals `exp(PMI)`, so the direct mode is essentially co-occurrence PMI with
support and lift thresholds. The context mode differs from 3.2 mainly in using confidence `P(b|a)` as the weight.

In [ ]:
from mlxtend.frequent_patterns import association_rules, fpgrowth


class AssociationRuleSubstitutes(SubstitutionModel):
    def __init__(self, mode="context", min_support=0.001, min_support_count=2,
                 min_lift=1.0, max_items=1000, max_recipes=200_000):
        assert mode in {"context", "direct"}
        self.mode = mode
        self.min_support, self.min_support_count = min_support, min_support_count
        self.min_lift, self.max_items, self.max_recipes = min_lift, max_items, max_recipes
        self.name = f"assoc_rules_{mode}"

    def fit(self, recipe_lists):
        if self.max_recipes and len(recipe_lists) > self.max_recipes:
            recipe_lists = random.Random(SEED).sample(recipe_lists, self.max_recipes)
        n = len(recipe_lists)
        counts = Counter(i for r in recipe_lists for i in set(r))
        items = sorted(w for w, _ in counts.most_common(self.max_items))
        X, self.vocab, self.idx = build_matrix(recipe_lists, vocab=items)

        onehot = pd.DataFrame.sparse.from_spmatrix(X.astype(bool), columns=self.vocab)
        min_sup = max(self.min_support, self.min_support_count / n)
        freq = fpgrowth(onehot, min_support=min_sup, use_colnames=True, max_len=2)

        kwargs = dict(metric="lift", min_threshold=self.min_lift)
        if "num_itemsets" in inspect.signature(association_rules).parameters:
            kwargs["num_itemsets"] = n   # required by some mlxtend versions
        rules = association_rules(freq, **kwargs)
        rules = rules[(rules["antecedents"].map(len) == 1) & (rules["consequents"].map(len) == 1)]
        self.rules = pd.DataFrame({
            "antecedent": rules["antecedents"].map(lambda s: next(iter(s))),
            "consequent": rules["consequents"].map(lambda s: next(iter(s))),
            "support": rules["support"], "confidence": rules["confidence"], "lift": rules["lift"],
        }).reset_index(drop=True)

        V = len(self.vocab)
        a = self.rules["antecedent"].map(self.idx).to_numpy()
        b = self.rules["consequent"].map(self.idx).to_numpy()
        self.lift = sparse.csr_matrix((self.rules["lift"].to_numpy(), (a, b)), shape=(V, V))
        conf = sparse.csr_matrix((self.rules["confidence"].to_numpy(), (a, b)), shape=(V, V))
        norms = np.sqrt(np.asarray(conf.multiply(conf).sum(axis=1)).ravel())
        norms[norms == 0] = 1.0
        self.profile = sparse.csr_matrix(sparse.diags(1.0 / norms) @ conf)
        return self

    def rank(self, query, k=10):
        q = self.idx[query]
        if self.mode == "direct":
            scores = self.lift[q].toarray().ravel()
        else:
            scores = (self.profile @ self.profile[q].T).toarray().ravel()
        return top_k(scores, self.vocab, k, exclude=q)

### 3.4 Word2Vec ingredient embeddings (gensim)

Each recipe is treated as a sentence. Ingredient order carries no meaning, so each recipe is shuffled and the window is set
to the longest recipe, which makes every ingredient a context for every other ingredient in the same recipe.
Results are only exactly reproducible with `workers=1`.

In [ ]:
from gensim.models import Word2Vec


class Word2VecSubstitutes(SubstitutionModel):
    name = "word2vec"

    def __init__(self, vector_size=100, window=None, min_count=5, sg=1, negative=10, epochs=10, workers=4):
        self.params = dict(vector_size=vector_size, min_count=min_count, sg=sg,
                           negative=negative, epochs=epochs, workers=workers, seed=SEED)
        self.window = window

    def fit(self, recipe_lists):
        rng = random.Random(SEED)
        sentences = [rng.sample(r, len(r)) for r in recipe_lists]
        window = self.window or max(len(s) for s in sentences)
        self.model = Word2Vec(sentences=sentences, window=window, **self.params)
        self.idx = self.model.wv.key_to_index
        return self

    def rank(self, query, k=10):
        return [(w, float(s)) for w, s in self.model.wv.most_similar(query, topn=k)]

## 4. Fit globally and per cuisine

In [ ]:
def make_models():
    return [
        PopularityBaseline(),
        CooccurrenceSimilarity(mode="context"),
        CooccurrenceSimilarity(mode="direct"),
        AssociationRuleSubstitutes(mode="context"),
        AssociationRuleSubstitutes(mode="direct"),
        Word2VecSubstitutes(),
    ]

def fit_all(recipe_lists):
    return {m.name: m.fit(recipe_lists) for m in make_models()}

all_lists = recipes["ingredients"].tolist()
global_models = fit_all(all_lists)

cuisine_sizes = recipes["cuisine"].value_counts()
cuisine_models = {}
for cuisine, size in cuisine_sizes.items():
    if size < MIN_RECIPES_PER_CUISINE:
        print(f"skip {cuisine}: only {size} recipes")
        continue
    cuisine_models[cuisine] = fit_all(recipes.loc[recipes["cuisine"] == cuisine, "ingredients"].tolist())
print("per-cuisine models:", sorted(cuisine_models))

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

In [ ]:
def compare(query, cuisines=("italian", "indian", "mexican"), methods=("cooccurrence_context", "word2vec"), k=5):
    rows = []
    for method in methods:
        scopes = [("global", global_models)] + [(c, cuisine_models[c]) for c in cuisines if c in cuisine_models]
        for scope, models in scopes:
            m = models[method]
            top = [w for w, _ in m.rank(query, k)] if m.knows(query) else ["(not in vocabulary)"]
            rows.append({"method": method, "scope": scope, f"top {k}": ", ".join(top)})
    return pd.DataFrame(rows)

compare("butter")

## 5. Evaluation (Hit@k, MRR)

The pairs below are **placeholders** to check that the pipeline runs. Replace them with the hand-built set by saving a CSV at
`data/substitution_pairs.csv` with columns `original, substitute` and optionally `cuisine`.
Pairs are passed through the same normaliser as the recipes. A pair whose ingredients are missing from a model's vocabulary
counts as a miss, so check the `coverage` column: the association-rule model only covers the `max_items` most frequent ingredients.

In [ ]:
PAIRS_CSV = DATA_DIR / "substitution_pairs.csv"
if PAIRS_CSV.exists():
    pairs = pd.read_csv(PAIRS_CSV)
else:
    pairs = pd.DataFrame([
        ("butter", "margarine"), ("sour cream", "greek yogurt"), ("shallot", "onion"),
        ("heavy cream", "half-and-half"), ("cilantro", "parsley"), ("honey", "maple syrup"),
        ("brown sugar", "sugar"), ("vegetable oil", "canola oil"), ("scallion", "green onion"),
        ("lime juice", "lemon juice"), ("ricotta cheese", "cottage cheese"), ("white wine", "chicken broth"),
    ], columns=["original", "substitute"])

pairs["original"] = pairs["original"].map(normalize_ingredient)
pairs["substitute"] = pairs["substitute"].map(normalize_ingredient)


def evaluate(model, pairs, ks=(1, 5, 10), k_max=50):
    ranks, covered = [], 0
    for o, s in zip(pairs["original"], pairs["substitute"]):
        if not (model.knows(o) and model.knows(s)):
            ranks.append(None)
            continue
        covered += 1
        cands = [w for w, _ in model.rank(o, k_max)]
        ranks.append(cands.index(s) + 1 if s in cands else None)
    res = {f"hit@{k}": np.mean([r is not None and r <= k for r in ranks]) for k in ks}
    res[f"mrr@{k_max}"] = np.mean([1 / r if r else 0.0 for r in ranks])
    res["coverage"] = covered / len(pairs)
    return res


rows = []
for scope, models in [("global", global_models)] + sorted(cuisine_models.items()):
    for name, m in models.items():
        rows.append({"scope": scope, "method": name, **evaluate(m, pairs)})
results = pd.DataFrame(rows)
results[results["scope"] == "global"].set_index("method").round(3)

In [ ]:
pivot = results[results["scope"] != "global"].pivot(index="scope", columns="method", values="hit@10")
ax = pivot.plot(kind="bar", figsize=(12, 4), width=0.8)
ax.set_ylabel("Hit@10")
ax.set_xlabel("cuisine model")
ax.set_title("Hit@10 by cuisine-specific model")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()